In [136]:
import pandas as pd
import numpy as np
from numpy import random

## Reading JSON file

In [192]:
df = pd.read_json("../data/auto.json")

In [193]:
pd.set_option('display.float_format', '{:.2f}'.format)

df

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.00,Ford,Focus
1,E432XX77RUS,1,6500.00,Toyota,Camry
2,7184TT36RUS,1,2100.00,Ford,Focus
3,X582HE161RUS,2,2000.00,Ford,Focus
4,92918M178RUS,1,5700.00,Ford,Focus
...,...,...,...,...,...
720,Y163O8161RUS,2,1600.00,Ford,Focus
721,M0309X197RUS,1,22300.00,Ford,Focus
722,O673E8197RUS,2,600.00,Ford,Focus
723,8610T8154RUS,1,2000.00,Ford,Focus


## Enrich the dataframe using a sample from that dataframe
* create a sample with 200 new observations with random_state = 21
* concatenate the sample with the initial dataframe to a new dataframe concat_rows

In [194]:
count_valid_rows = 0 
concat_rows = pd.concat([df])
while  concat_rows.shape[0] != (df.shape[0] + 200):
    new_sample = df.sample(n=200 - count_valid_rows, random_state=21, ignore_index=True)
    valid = df[["CarNumber", "Make", "Model"]].drop_duplicates()
    new_sample = new_sample.merge(valid, how='inner', on=['CarNumber', 'Make', 'Model'])
    count_valid_rows = new_sample.shape[0]
    concat_rows = pd.concat([concat_rows, new_sample], ignore_index=True)

concat_rows


,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.00,Ford,Focus
1,E432XX77RUS,1,6500.00,Toyota,Camry
2,7184TT36RUS,1,2100.00,Ford,Focus
3,X582HE161RUS,2,2000.00,Ford,Focus
4,92918M178RUS,1,5700.00,Ford,Focus
...,...,...,...,...,...
920,8182XX154RUS,1,200.00,Ford,Focus
921,X796TH96RUS,1,500.00,Ford,Focus
922,T011MY163RUS,2,4000.00,Ford,Focus
923,T341CC96RUS,2,1000.00,Volkswagen,Passat


## Enrich the dataframe concat_rows by a new column with the data generated
* create a series with the name Year using random integers from 1980 to 2019
* use np.random.seed(21) before generating the years
* concatenate the series with the dataframe and name it fines

In [195]:
np.random.seed(21)
years = np.random.randint(1980, 2019, concat_rows.shape[0])
Year = pd.Series(years)

fines = pd.concat([concat_rows, Year], axis=1)
fines.rename(columns={0: "Year"}, inplace=True)

fines


,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995
2,7184TT36RUS,1,2100.00,Ford,Focus,1984
3,X582HE161RUS,2,2000.00,Ford,Focus,2015
4,92918M178RUS,1,5700.00,Ford,Focus,2014
...,...,...,...,...,...,...
920,8182XX154RUS,1,200.00,Ford,Focus,1996
921,X796TH96RUS,1,500.00,Ford,Focus,2002
922,T011MY163RUS,2,4000.00,Ford,Focus,1996
923,T341CC96RUS,2,1000.00,Volkswagen,Passat,2012


## Enrich the dataframe with the data from another dataframe
* create a new dataframe with the car numbers and their owners
* append 5 more observations to the fines dataframe 
* delete the dataframe last 20 observations from the owners and add 3 new observations 
* join both dataframes

In [196]:
surname = pd.read_json("../../datasets/surname.json")
surname = surname[1:]
surname.rename(columns={0: "NAME", 1: "COUNT", 2: "RANK"}, inplace=True)

only_surnames = surname["NAME"]

surname

,NAME,COUNT,RANK
1,ADAMS,427865,42
2,ALLEN,482607,33
3,ALVAREZ,233983,92
4,ANDERSON,784404,15
5,BAILEY,277845,72
...,...,...,...
96,WILLIAMS,1625252,3
97,WILSON,801882,14
98,WOOD,250715,84
99,WRIGHT,458980,35


In [197]:
UniqueCarNum = fines["CarNumber"].unique()
Repeated_Surnames = only_surnames.sample(n=len(UniqueCarNum), random_state=21, replace=True, ignore_index=True)

owners = pd.DataFrame()
owners["CarNumber"] = UniqueCarNum 
owners["SURNAME"] = Repeated_Surnames

owners

,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ
...,...,...
526,O136HO197RUS,CAMPBELL
527,O22097197RUS,HALL
528,M0309X197RUS,BAKER
529,O673E8197RUS,DIAZ


In [198]:
new_rows = pd.DataFrame([{"CarNumber": "123456789RUS", "Refund": 2, "Fines": 3000, "Make": "Ford", "Model": "Focus", "Year": 1980},{"CarNumber": "ABVJS@#&$*JWRUS", "Refund": 2, "Fines": 30, "Make": "Audi", "Model": "Focus", "Year": 2010}, {"CarNumber": "ABCD1234RUS", "Refund": 1, "Fines": 1000, "Make": "Toyota", "Model": "Good", "Year": 2000}, {"CarNumber": "HAJR234AJGORUS", "Refund": 3, "Fines": 1500, "Make": "Ford", "Model": "Focus", "Year": 2005}, {"CarNumber": "SJDIF23GJKS34RUS", "Refund": 3, "Fines": 10000, "Make": "Toyota", "Model": "Focus", "Year": 2015}])
fines = pd.concat([fines, new_rows], ignore_index=True)
fines

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995
2,7184TT36RUS,1,2100.00,Ford,Focus,1984
3,X582HE161RUS,2,2000.00,Ford,Focus,2015
4,92918M178RUS,1,5700.00,Ford,Focus,2014
...,...,...,...,...,...,...
925,123456789RUS,2,3000.00,Ford,Focus,1980
926,ABVJS@#&$*JWRUS,2,30.00,Audi,Focus,2010
927,ABCD1234RUS,1,1000.00,Toyota,Good,2000
928,HAJR234AJGORUS,3,1500.00,Ford,Focus,2005


In [199]:
owners = owners.drop(owners.index[-20:])
three_new_rows = pd.DataFrame([{"CarNumber": "HSDK4738SDJFJRUS", "SURNAME": "CAMPBELL"}, {"CarNumber": "LJKSD36SJDF38RUS", "SURNAME": "ROSS"}, {"CarNumber": "AO23SDFJ8SJRUS", "SURNAME": "MORGAN"}])
owners = pd.concat([owners, three_new_rows], ignore_index=True)

owners

,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ
...,...,...
509,O50197197RUS,WRIGHT
510,7608EE777RUS,HILL
511,HSDK4738SDJFJRUS,CAMPBELL
512,LJKSD36SJDF38RUS,ROSS


In [200]:
#only the car numbers that exist in both dataframes
only_in_both = pd.merge(owners, fines, how="inner", on=["CarNumber"])

only_in_both 

,CarNumber,SURNAME,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,RICHARDSON,2,3200.00,Ford,Focus,1989
1,Y163O8161RUS,RICHARDSON,2,1600.00,Ford,Focus,1999
2,E432XX77RUS,ROSS,1,6500.00,Toyota,Camry,1995
3,E432XX77RUS,ROSS,2,13000.00,Toyota,Camry,1992
4,7184TT36RUS,MORGAN,1,2100.00,Ford,Focus,1984
...,...,...,...,...,...,...,...
894,E41977152RUS,BAKER,2,2400.00,Ford,Focus,2001
895,9464EX178RUS,MARTIN,2,2100.00,Ford,Focus,1993
896,O50197197RUS,WRIGHT,2,7800.00,Ford,Focus,1986
897,7608EE777RUS,HILL,1,4000.00,Skoda,Octavia,2013


In [201]:
#all the car numbers that exist in both dataframes
all_in_both = pd.merge(owners, fines, how="outer", on=["CarNumber"])

all_in_both 

,CarNumber,SURNAME,Refund,Fines,Make,Model,Year
0,123456789RUS,NaN,2.00,3000.00,Ford,Focus,1980.00
1,704687163RUS,ADAMS,2.00,1400.00,Ford,Focus,2014.00
2,704787163RUS,MORGAN,2.00,2800.00,Ford,Focus,2005.00
3,704987163RUS,MITCHELL,2.00,8594.59,Ford,Focus,2014.00
4,705287163RUS,GOMEZ,2.00,2000.00,Ford,Focus,1990.00
...,...,...,...,...,...,...,...
928,Y973O8197RUS,YOUNG,2.00,8594.59,Ford,Focus,2005.00
929,Y973O8197RUS,YOUNG,1.00,34800.00,Ford,Focus,2013.00
930,Y973O8197RUS,YOUNG,1.00,69600.00,Ford,Focus,1989.00
931,Y973O8197RUS,YOUNG,1.00,34800.00,Ford,Focus,2009.00


In [202]:
#only the car numbers from the fines dataframe
only_from_fines = pd.merge(owners, fines, how="right", on=["CarNumber"])

only_from_fines

,CarNumber,SURNAME,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,RICHARDSON,2,3200.00,Ford,Focus,1989
1,E432XX77RUS,ROSS,1,6500.00,Toyota,Camry,1995
2,7184TT36RUS,MORGAN,1,2100.00,Ford,Focus,1984
3,X582HE161RUS,BAILEY,2,2000.00,Ford,Focus,2015
4,92918M178RUS,LOPEZ,1,5700.00,Ford,Focus,2014
...,...,...,...,...,...,...,...
925,123456789RUS,NaN,2,3000.00,Ford,Focus,1980
926,ABVJS@#&$*JWRUS,NaN,2,30.00,Audi,Focus,2010
927,ABCD1234RUS,NaN,1,1000.00,Toyota,Good,2000
928,HAJR234AJGORUS,NaN,3,1500.00,Ford,Focus,2005


In [203]:
#only the car numbers from the owners dataframe
only_from_owners= pd.merge(owners, fines, how="left", on=["CarNumber"])

only_from_owners

,CarNumber,SURNAME,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,RICHARDSON,2.00,3200.00,Ford,Focus,1989.00
1,Y163O8161RUS,RICHARDSON,2.00,1600.00,Ford,Focus,1999.00
2,E432XX77RUS,ROSS,1.00,6500.00,Toyota,Camry,1995.00
3,E432XX77RUS,ROSS,2.00,13000.00,Toyota,Camry,1992.00
4,7184TT36RUS,MORGAN,1.00,2100.00,Ford,Focus,1984.00
...,...,...,...,...,...,...,...
897,7608EE777RUS,HILL,1.00,4000.00,Skoda,Octavia,2013.00
898,7608EE777RUS,HILL,1.00,4000.00,Skoda,Octavia,1987.00
899,HSDK4738SDJFJRUS,CAMPBELL,NaN,NaN,NaN,NaN,NaN
900,LJKSD36SJDF38RUS,ROSS,NaN,NaN,NaN,NaN,NaN


## Creation a pivot table from the fines dataframe

In [205]:

PivotTable = pd.pivot_table(fines, index=["Make", "Model"], columns=["Year"], values=["Fines"], aggfunc=[sum])

PivotTable


/var/folders/jb/zkfyxcwx1_z7_qyw8llwwzxw0000gn/T/ipykernel_2400/1856106521.py:1: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  PivotTable = pd.pivot_table(fines, index=["Make", "Model"], columns=["Year"], values=["Fines"], aggfunc=[sum])


sum                                                    \
                      Fines                                                     
Year                   1980      1981      1982      1983      1984      1985   
Make       Model                                                                
Audi       Focus        NaN       NaN       NaN       NaN       NaN       NaN   
           NaN          NaN       NaN       NaN       NaN       NaN       NaN   
BMW        NaN          NaN       NaN       NaN       NaN       NaN       NaN   
Ford       Focus   92194.59 266783.76 107283.76 147289.17 106000.00 307494.59   
           Mondeo       NaN       NaN  46200.00       NaN       NaN       NaN   
Skoda      Octavia 13794.59   1900.00   8894.59       NaN   1300.00 153594.59   
Toyota     Camry   12000.00       NaN   1000.00   8594.59   1000.00       NaN   
           Corolla      NaN   6800.00       NaN  12800.00       NaN   4400.00   
           Focus        NaN       NaN       NaN       NaN       NaN       NaN   
           Good         NaN       NaN       NaN       NaN       NaN       NaN   
Volkswagen Golf    20800.00   8594.59   5000.00    200.00       NaN 168000.00   
           Jetta        NaN   1000.00       NaN       NaN       NaN   9000.00   
           NaN      1300.00   7900.00       NaN       NaN       NaN       NaN   
           Passat    900.00  12500.00       NaN   1100.00   8594.59       NaN   
           Touareg      NaN       NaN       NaN       NaN       NaN       NaN   
Volvo      NaN          NaN       NaN       NaN       NaN       NaN       NaN   

                                                         ...            \
                                                         ...             
Year                   1986     1987     1988      1989  ...      2009   
Make       Model                                         ...             
Audi       Focus        NaN      NaN      NaN       NaN  ...       NaN   
           NaN          NaN      NaN      NaN       NaN  ...       NaN   
BMW        NaN          NaN      NaN      NaN       NaN  ...       NaN   
Ford       Focus   69700.00 98189.17 69667.52 200889.17  ... 159894.59   
           Mondeo       NaN      NaN      NaN       NaN  ...       NaN   
Skoda      Octavia      NaN  6000.00  5100.00   8594.59  ...       NaN   
Toyota     Camry   19800.00      NaN      NaN    800.00  ...       NaN   
           Corolla      NaN 54300.00      NaN   7800.00  ...   8594.59   
           Focus        NaN      NaN      NaN       NaN  ...       NaN   
           Good         NaN      NaN      NaN       NaN  ...       NaN   
Volkswagen Golf         NaN   300.00      NaN    300.00  ...       NaN   
           Jetta        NaN      NaN 46000.00   4000.00  ...       NaN   
           NaN      7400.00      NaN      NaN       NaN  ...       NaN   
           Passat  16000.00  2000.00  8594.59       NaN  ...   3200.00   
           Touareg      NaN      NaN      NaN       NaN  ...   5800.00   
Volvo      NaN          NaN      NaN      NaN       NaN  ...   6800.00   

                                                                              \
                                                                               
Year                   2010      2011      2012      2013     2014      2015   
Make       Model                                                               
Audi       Focus      30.00       NaN       NaN       NaN      NaN       NaN   
           NaN          NaN       NaN       NaN       NaN      NaN       NaN   
BMW        NaN          NaN   8594.59       NaN       NaN      NaN       NaN   
Ford       Focus   96000.00 117194.59 152989.17 297378.35 90378.35 172700.00   
           Mondeo       NaN       NaN       NaN  41100.00      NaN       NaN   
Skoda      Octavia  3000.00   3000.00   1700.00  11800.00 18900.00  16394.59   
Toyota     Camry   22400.00       NaN   7500.00       NaN      NaN       NaN   
           Corolla  6000.00   3400.00     

## Saving both the fines and owners dataframes to CSV files without an index

In [206]:
fines.to_csv("../data/fines.csv", index=False)
owners.to_csv("../data/owners.csv", index=False)